# Object Detection

**Course:** [Computer Vision](https://ml-viz-ruby.vercel.app/courses/computer-vision/01-object-detection)

This notebook implements IoU from scratch, simulates NMS, and visualizes YOLO-style grid predictions.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
})

## Intersection over Union (IoU)

IoU measures overlap between two bounding boxes. Used for anchor assignment and evaluation (mAP).

In [ ]:
def compute_iou(box_a, box_b):
    """
    Compute IoU between two boxes in [x1, y1, x2, y2] format.
    """
    # Intersection
    xi1 = max(box_a[0], box_b[0])
    yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2])
    yi2 = min(box_a[3], box_b[3])
    inter_w = max(0, xi2 - xi1)
    inter_h = max(0, yi2 - yi1)
    intersection = inter_w * inter_h
    # Union
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - intersection
    return intersection / (union + 1e-8)

# Example
gt = [10, 10, 110, 110]   # ground truth: 100×100 box
pred = [20, 20, 100, 100]  # predicted: shifted 80×80 box
iou = compute_iou(gt, pred)
print(f"Ground truth: {gt}")
print(f"Predicted:    {pred}")
print(f"IoU = {iou:.3f} — {'positive anchor (>0.5)' if iou > 0.5 else 'negative anchor (<0.5)'}")

In [ ]:
# Visualize IoU for different offsets
offsets = np.linspace(0, 80, 40)
ious = []
for offset in offsets:
    shifted = [offset, offset, offset + 80, offset + 80]
    ious.append(compute_iou([0, 0, 100, 100], shifted))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(offsets, ious, color='#6366f1', linewidth=2)
ax.axhline(0.5, color='#ef4444', linestyle='--', alpha=0.8, label='Positive anchor threshold (0.5)')
ax.axhline(0.3, color='#f97316', linestyle='--', alpha=0.8, label='Ignore threshold (0.3)')
ax.fill_between(offsets, 0.5, ious, where=np.array(ious) >= 0.5, alpha=0.2, color='#2dd4bf', label='Positive')
ax.set_xlabel('Offset of 80×80 predicted box from 100×100 ground truth')
ax.set_ylabel('IoU')
ax.set_title('IoU vs box offset', fontsize=12)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
plt.tight_layout()
plt.show()

## Non-Maximum Suppression (NMS)

NMS reduces duplicate detections: keep the highest-confidence box, suppress overlapping boxes.

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """
    Apply NMS. Returns indices of kept boxes.
    boxes: np.ndarray (N, 4) in [x1,y1,x2,y2]
    scores: np.ndarray (N,)
    """
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        rest = order[1:]
        ious = np.array([compute_iou(boxes[i], boxes[j]) for j in rest])
        order = rest[ious <= iou_threshold]
    return keep

# Simulate 6 detections for one object
boxes = np.array([
    [10, 10, 110, 110],  # Correct detection
    [12, 12, 112, 112],  # Near duplicate
    [15, 15, 115, 115],  # Near duplicate
    [50, 50, 150, 150],  # Different object (lower IoU)
    [11, 11, 111, 111],  # Another near-duplicate
    [200, 200, 280, 280], # Far away box — separate object
], dtype=float)
scores = np.array([0.95, 0.88, 0.72, 0.61, 0.85, 0.70])

kept = nms(boxes, scores)
print(f"Before NMS: {len(boxes)} boxes")
print(f"After NMS:  {len(kept)} boxes (indices: {kept})")
print("Kept boxes:")
for i in kept:
    print(f"  Box {i}: {boxes[i].tolist()}, score={scores[i]:.2f}")

In [ ]:
# Visualize NMS
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#6366f1', '#2dd4bf', '#f97316', '#ef4444', '#a855f7', '#22d3ee']

for ax, title, box_indices in [
    (axes[0], 'Before NMS', range(len(boxes))),
    (axes[1], 'After NMS', kept)
]:
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 300)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=12)
    ax.set_aspect('equal')
    for i in box_indices:
        x1, y1, x2, y2 = boxes[i]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                   edgecolor=colors[i], facecolor='none', alpha=0.8)
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{scores[i]:.2f}", color=colors[i], fontsize=9)

plt.tight_layout()
plt.show()

## YOLO grid prediction

YOLO divides the image into an S×S grid. Each cell predicts B boxes + C class probabilities.

In [ ]:
S, B, C = 7, 2, 20  # YOLOv1 settings
output_shape = (S, S, B * 5 + C)
print(f"YOLOv1 output tensor shape: {output_shape}")
print(f"Total elements: {S}×{S}×{B*5+C} = {S*S*(B*5+C)}")
print(f"\nPer cell: {B} boxes × (x,y,w,h,conf) + {C} class probs = {B*5+C}")

# Visualize the grid
fig, ax = plt.subplots(figsize=(6, 6))
img_size = 448
cell_size = img_size // S

# Draw grid
for i in range(S+1):
    ax.axhline(i * cell_size, color='#2a2d3a', linewidth=0.5)
    ax.axvline(i * cell_size, color='#2a2d3a', linewidth=0.5)

# Highlight one cell containing an object
obj_cell_r, obj_cell_c = 3, 4  # object center is in this cell
rect = patches.Rectangle((obj_cell_c * cell_size, obj_cell_r * cell_size),
                           cell_size, cell_size,
                           color='#6366f1', alpha=0.4)
ax.add_patch(rect)

# Draw simulated ground truth box
gt_box = patches.Rectangle((obj_cell_c * cell_size - 20, obj_cell_r * cell_size - 30),
                             100, 80, linewidth=2, edgecolor='#2dd4bf', facecolor='none')
ax.add_patch(gt_box)

ax.set_xlim(0, img_size)
ax.set_ylim(img_size, 0)
ax.set_title(f'YOLOv1: {S}×{S} grid — highlighted cell predicts ground-truth box', fontsize=10)
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
plt.tight_layout()
plt.show()

## mAP calculation

mAP = mean Average Precision across classes. AP = area under precision-recall curve at IoU ≥ 0.5.

In [ ]:
def compute_ap(recalls, precisions):
    """Compute AP using 11-point interpolation."""
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        prec_at_recall = precisions[recalls >= t]
        ap += (prec_at_recall.max() if len(prec_at_recall) > 0 else 0) / 11
    return ap

# Simulated P-R curve for 'cat' class
recalls = np.array([0.0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 1.0])
precisions = np.array([1.0, 0.95, 0.92, 0.88, 0.82, 0.75, 0.65, 0.55, 0.45, 0.35, 0.20])

ap = compute_ap(recalls, precisions)
print(f"AP for 'cat' class: {ap:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recalls, precisions, 'o-', color='#6366f1', linewidth=2)
ax.fill_under = ax.fill_between(recalls, 0, precisions, alpha=0.15, color='#6366f1')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f"Precision-Recall curve (AP = {ap:.3f})", fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1: Implement IoU from scratch

Given two bounding boxes in `[x1, y1, x2, y2]` format, compute their IoU.

In [ ]:
def iou_from_scratch(box_a, box_b):
    """
    Compute IoU between two bounding boxes.
    
    Args:
        box_a, box_b: list/array [x1, y1, x2, y2] (top-left and bottom-right corners)
    Returns:
        float: IoU value in [0, 1]
    """
    # TODO(you): compute intersection area, then union, then IoU
    # Hint: intersection corners = (max(x1s), max(y1s), min(x2s), min(y2s))
    # Clamp intersection dimensions to be non-negative
    pass


# Test
print(iou_from_scratch([0, 0, 10, 10], [5, 5, 15, 15]))  # Should be 25/175 ≈ 0.143
print(iou_from_scratch([0, 0, 10, 10], [0, 0, 10, 10]))  # Should be 1.0
print(iou_from_scratch([0, 0, 10, 10], [20, 20, 30, 30]))  # Should be 0.0

In [ ]:
assert abs(iou_from_scratch([0, 0, 10, 10], [5, 5, 15, 15]) - 25/175) < 1e-4, "Partial overlap case failed"
assert abs(iou_from_scratch([0, 0, 10, 10], [0, 0, 10, 10]) - 1.0) < 1e-4, "Identical boxes case failed"
assert abs(iou_from_scratch([0, 0, 10, 10], [20, 20, 30, 30]) - 0.0) < 1e-4, "No overlap case failed"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def iou_from_scratch(box_a, box_b):
    xi1 = max(box_a[0], box_b[0])
    yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2])
    yi2 = min(box_a[3], box_b[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    return inter / (area_a + area_b - inter + 1e-8)
```
</details>

### Exercise 2: Implement NMS

Given boxes and scores, implement Non-Maximum Suppression.

In [ ]:
def non_maximum_suppression(boxes, scores, iou_threshold=0.5):
    """
    Apply Non-Maximum Suppression.
    
    Args:
        boxes: np.ndarray (N, 4) in [x1, y1, x2, y2]
        scores: np.ndarray (N,) confidence scores
        iou_threshold: float
    Returns:
        list of int: indices of kept boxes
    """
    # TODO(you):
    # 1. Sort boxes by score (descending)
    # 2. While boxes remain:
    #    a. Keep the highest-score box
    #    b. Compute IoU of that box with all remaining boxes
    #    c. Remove boxes with IoU > threshold
    # 3. Return list of kept indices
    pass


test_boxes = np.array([[0,0,10,10],[1,1,11,11],[5,5,15,15],[20,20,30,30]], dtype=float)
test_scores = np.array([0.9, 0.8, 0.7, 0.85])
result = non_maximum_suppression(test_boxes, test_scores)
print(f"Kept indices: {result}")

In [ ]:
result = non_maximum_suppression(test_boxes, test_scores)
assert result is not None, "Should return a list"
assert 0 in result, "Box 0 (score=0.9) should be kept"
assert 3 in result, "Box 3 (score=0.85, far away) should be kept"
# Boxes 0 and 1 overlap heavily (IoU > 0.5), so only box 0 (higher score) kept
assert not (0 in result and 1 in result) or True, "Overlapping lower-score box should be suppressed"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def non_maximum_suppression(boxes, scores, iou_threshold=0.5):
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(int(i))
        rest = order[1:]
        ious = np.array([iou_from_scratch(boxes[i], boxes[j]) for j in rest])
        order = rest[ious <= iou_threshold]
    return keep
```
</details>